# 02. 전이학습 — 사전학습 모델 활용하기

ImageNet으로 학습된 **MobileNetV2** 를 가져와 Fashion-MNIST에 맞게 재사용합니다.
대규모 데이터로 학습된 특징을 빌려오면, 적은 학습으로 높은 정확도를 얻을 수 있습니다 — 이것이 **전이학습**입니다.

흐름:
1. **데이터 준비** — Fashion-MNIST를 MobileNetV2 입력 형식(3채널·96×96)에 맞게 변환
2. **데이터 증강** — Keras 전처리 레이어로 증강
3. **특징 추출(feature extraction)** — 사전학습 가중치는 고정, 분류기만 학습
4. **미세조정(fine-tuning)** — 상위 층을 낮은 학습률로 추가 학습

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt

plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False
print('TensorFlow', tf.__version__)

## 1. 데이터 준비

Fashion-MNIST(28×28 흑백)를 MobileNetV2가 받을 수 있는 형식으로 변환합니다.
흑백을 3채널로 복제하고 최소 입력 크기(96×96)로 키운 뒤, `tf.data` 파이프라인으로 공급하고 MobileNetV2 전용 전처리(`preprocess_input`)로 픽셀을 [-1,1] 범위로 맞춥니다.

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()
class_names = ['티셔츠', '바지', '풀오버', '드레스', '코트',
               '샌들', '셔츠', '스니커즈', '가방', '앵클부츠']

IMG = 96
preprocess = keras.applications.mobilenet_v2.preprocess_input

def make_ds(x, y, training=False):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if training:
        ds = ds.shuffle(2000)
    def prep(im, lb):
        im = tf.cast(im, tf.float32)
        im = tf.expand_dims(im, -1)            # (28,28) -> (28,28,1)
        im = tf.image.grayscale_to_rgb(im)     # 흑백 -> 3채널
        im = tf.image.resize(im, (IMG, IMG))   # -> 96x96
        return preprocess(im), lb              # MobileNetV2 전처리
    return ds.map(prep, num_parallel_calls=tf.data.AUTOTUNE).batch(128).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(x_train, y_train, training=True)
test_ds = make_ds(x_test, y_test)
print('데이터 준비 완료')

## 2. 데이터 증강 레이어

Keras는 증강을 **모델의 일부 레이어**로 넣을 수 있습니다. 학습 시에만 무작위로 적용되고, 추론 시에는 비활성화됩니다.

In [ ]:
data_augmentation = keras.Sequential([
    keras.layers.RandomFlip('horizontal'),
    keras.layers.RandomRotation(0.1),
], name='data_augmentation')
print(data_augmentation)

## 3. 특징 추출 (feature extraction)

사전학습된 MobileNetV2를 불러오되 분류기(`include_top=False`)는 제외하고, 가중치를 **고정(freeze)** 합니다.
그 위에 Fashion-MNIST용 새 분류기를 얹어, 새로 추가한 층만 학습합니다. Functional API로 구성합니다.

In [ ]:
base_model = keras.applications.MobileNetV2(
    input_shape=(IMG, IMG, 3), include_top=False, weights='imagenet')
base_model.trainable = False  # 사전학습 가중치 고정

inputs = keras.Input(shape=(IMG, IMG, 3))
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dropout(0.2)(x)
outputs = keras.layers.Dense(10, activation='softmax')(x)
model = keras.Model(inputs, outputs)

model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
hist1 = model.fit(train_ds, validation_data=test_ds, epochs=3, verbose=2)

## 4. 미세조정 (fine-tuning)

특징 추출로 분류기가 어느 정도 학습된 뒤, base_model의 **상위 일부 층을 풀어** 낮은 학습률로 함께 학습하면 정확도가 더 오릅니다.

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-30]:   # 상위 30개 층만 학습
    layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(1e-5),  # 낮은 학습률
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
hist2 = model.fit(train_ds, validation_data=test_ds, epochs=2, verbose=2)

In [ ]:
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f'전이학습 최종 테스트 정확도: {test_acc*100:.2f}%')

### 정리

- ImageNet 사전학습 MobileNetV2를 freeze해 **특징 추출**로 분류기를 먼저 학습하고, 이후 상위 층을 풀어 **미세조정**했습니다.
- 데이터 증강을 모델 레이어로 통합하는 Keras 방식을 사용했습니다.
- 처음부터 학습하는 것보다 적은 에폭으로 높은 정확도를 얻습니다 — 전이학습의 핵심입니다.